<a href="https://colab.research.google.com/github/ans125na/project/blob/main/Project_Omar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# مشروع: مساعد خدمات بطاقة الهوية الوطنية الذكي (National ID Assistant)
# الاسم: عمر مانع سعود الحربي | التخصص: الأمن السيبراني
# ==============================================================================

import re
from dataclasses import dataclass

print("=" * 60)
print("مشروع: مساعد خدمات بطاقة الهوية الوطنية الذكي (National ID Assistant)")
print("الاسم: عمر مانع سعود الحربي | التخصص: الأمن السيبراني")
print("=" * 60)

# ------------------------------------------------------------------------------
# SECTION 0: إعدادات المشروع وحماية البيانات (PROJECT SETTINGS & GUARDRAILS)
# ------------------------------------------------------------------------------

ASSISTANT_NAME = "المساعد الذكي - الهوية الوطنية"
CURRENT_USER = "عمر الحربي"

# الخدمات المتاحة
SERVICES = [
    "تجديد بطاقة الهوية الوطنية",
  "اصدار بطاقة هوية بدل تالف",
   "تغيير البيانات الشخصية",
   "تعديل الحالة الاجتماعية"
]

# الكلمات المفتاحية المقبولة في النطاق
SCOPE_WORDS = [
    "تجديد", "هوية", "بطاقة", "وطنية", "أحوال",
   "بدل", "تالف", "مفقود", "تعديل", "حالة", "إصدار"
]

# العبارات المحظورة (Prompt Injection Guard)
BLOCKED_PHRASES = [
    "ignore previous instructions",
    "system prompt",
   "تجاهل التعليمات",
   "تجاهل الأوامر"
]

# أنماط البيانات الحساسة (PII Patterns)
PII_PATTERNS = {
    "NATIONAL_ID": r"\b1\d{9}\b",
    "PHONE": r"\b05\d{8}\b"
}

# الردود القياسية
REFUSALS = {
    "injection": "عذراً، لا يمكنني الاستجابة لهذه التعليمات. أنا مساعد خاص بخدمات الهوية الوطنية والأحوال المدنية فقط.",
    "off_scope": "عذراً، هذا السؤال خارج نطاق اختصاصي. أنا مساعد متخصص في خدمات الهوية الوطنية والأحوال المدنية فقط.",
    "leak": "عذراً، لا يمكنني عرض هذه المعلومات الحساسة."
}

PRICE_PER_TOKEN = {"flagship": 10, "cheap": 2}
LATENCY = {"flagship": 1.2, "cheap": 0.4}

print(f"تم تحميل إعدادات المشروع وحماية البيانات لـ: {ASSISTANT_NAME}")

# ------------------------------------------------------------------------------
# SECTION 1: الواجهة والمحول (INTERFACE & ADAPTERS)
# ------------------------------------------------------------------------------

@dataclass
class LLMRequest:
    messages: list
    model: str = "demo-model"
    max_tokens: int = 512

@dataclass
class LLMResponse:
    text: str

class OpenAICompatClient:
    def __init__(self, name="OpenAI"):
        self.name = name

    def complete(self, request: LLMRequest) -> LLMResponse:
        question = request.messages[-1]["content"]

       # فحص محاولات حقن الأوامر (Prompt Injection)
        for phrase in BLOCKED_PHRASES:
            if phrase.lower() in question.lower():
                return LLMResponse(text=REFUSALS["injection"])

       # فحص الخروج عن النطاق (Off-scope)
        if not any(word in question for word in SCOPE_WORDS):
            return LLMResponse(text=REFUSALS["off_scope"])

        return LLMResponse(text=f"[{self.name}] تم استلام استفسارك بنجاح بخصوص '{question}'. كيف يمكنني مساعدتك أكثر؟")

# ------------------------------------------------------------------------------
# SECTION 2: سجل الأدوات وبوابة الصلاحيات (TOOL REGISTRY & AUTHORIZATION GATE)
# ------------------------------------------------------------------------------

APPLICATIONS = {
    "طلب تجديد - قيد المعالجة": "1118564937",
    "طلب إصدار بدل تالف - مكتمل": "1012345678"
}

def check_id_status(national_id: str):
    for status, nid in APPLICATIONS.items():
        if nid == national_id:
            return f"حالة الطلب لرقم الهوية ({national_id}): {status}"
    return "لا يوجد طلبات مسجلة برقم الهوية هذا."

def book_appointment(name: str, city: str, date: str):
    return f"تم حجز موعد لـ {name} في مدينة {city} بتاريخ {date} بنجاح."

TOOLS = {
    "check_id_status": {"function": check_id_status, "risk": "read_only"},
    "book_appointment": {"function": book_appointment, "risk": "side_effecting"}
}

def tool_run(name: str, args: dict):
    if name not in TOOLS:
        return "خطأ: الأداة غير موجودة."

    tool = TOOLS[name]

    # بوابة الصلاحيات (Authorization Gate)
    if tool["risk"] == "side_effecting" and args.get("name") != CURRENT_USER:
        return "تنبيه: غير مصرح لك بتنفيذ هذا العمل لنص مستخدم آخر!"

    return tool["function"](**args)

# ------------------------------------------------------------------------------
# SECTION 3: التقييم والتشغيل (EVALUATION & RUN)
# ------------------------------------------------------------------------------

GOLDEN_SET = [
    {"question": "كيف أجدد بطاقة الهوية الوطنية؟", "expected": "استجابة مقبولة"},
    {"question": "ما هي أفضل طريقة لطبخ الكبسة؟", "expected": "خارج النطاق"}
]

if __name__ == "__main__":
    print("\n--- تشغيل تجريبي لأدوات النظام والتحقق من الصلاحيات ---")

    # 1. الاستعلام عن حالة الهوية (Read-only tool)
    res1 = tool_run("check_id_status", {"national_id": "1118564937"})
    print(f"1. نتيجة الاستعلام: {res1}")

  # 2. حجز موعد للمستخدم الحالي (Authorized action)
    res2 = tool_run("book_appointment", {"name": CURRENT_USER, "city": "الرياض", "date": "2026-10-15"})
    print(f"2. نتيجة حجز الموعد المصرح: {res2}")

   # 3. محاولة حجز موعد لمستخدم آخر (Unauthorized action)
    res3 = tool_run("book_appointment", {"name": "علي صالح", "city": "الرياض", "date": "2026-10-15"})
    print(f"3. نتيجة حجز موعد غير مصرح: {res3}")

مشروع: مساعد خدمات بطاقة الهوية الوطنية الذكي (National ID Assistant)
الاسم: عمر مانع سعود الحربي | التخصص: الأمن السيبراني
تم تحميل إعدادات المشروع وحماية البيانات لـ: المساعد الذكي - الهوية الوطنية

--- تشغيل تجريبي لأدوات النظام والتحقق من الصلاحيات ---
1. نتيجة الاستعلام: حالة الطلب لرقم الهوية (1118564937): طلب تجديد - قيد المعالجة
2. نتيجة حجز الموعد المصرح: تم حجز موعد لـ عمر الحربي في مدينة الرياض بتاريخ 2026-10-15 بنجاح.
3. نتيجة حجز موعد غير مصرح: تنبيه: غير مصرح لك بتنفيذ هذا العمل لنص مستخدم آخر!
